# YOLOv11 Chess Pieces Detection - Training (YOLO Format)

Notebook này hỗ trợ huấn luyện mô hình YOLOv11m trên cả **Google Colab** và **Kaggle** sử dụng định dạng **YOLO Dataset Format**.


## Step 0 - Environment & Path Setup

In [ ]:
# ============================================================
# Minimalist Environment & Path Configuration
# ============================================================
from pathlib import Path
import os

# Set environment manually: "colab" or "kaggle"
ENV = "colab"

if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

    # Project root directory on Google Drive
    DRIVE_PROJECT = Path("/content/drive/MyDrive/chess_pieces_detection")
    LOCAL_DIR = Path("/content/datasets")
    RUNS_DIR = str(DRIVE_PROJECT / "runs")
    ZIP_DRIVE_PATH = DRIVE_PROJECT / "datasets" / "chessred_yolo_format.zip"

elif ENV == "kaggle":
    LOCAL_DIR = Path("/kaggle/working/datasets")
    RUNS_DIR = "/kaggle/working/runs"
    ZIP_DRIVE_PATH = None

LOCAL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Environment set to: {ENV.upper()}")
print(f"LOCAL_DIR : {LOCAL_DIR}")
print(f"RUNS_DIR  : {RUNS_DIR}")


Detected environment: KAGGLE
  LOCAL_DIR  : /kaggle/working/datasets
  INPUT_DIR  : /kaggle/input
  RUNS_DIR   : /kaggle/working/runs
  RESUME_CKPT   : yolo11m.pt

[Kaggle] Outputs saved to /kaggle/working/ -- commit via
         'Save Version' -> 'Save & Run All' to persist them.

Path configuration complete.


## Step 1: Install Dependencies
Install all required Python libraries for training.

In [ ]:
# Install required libraries
!pip install -q ultralytics gdown pyyaml


Installing project dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 4.9 MB/s eta 0:00:00


## Step 2: Dataset Preparation
To avoid downloading the heavy dataset from the internet every time the runtime restarts, this step will:
1. **Colab**: Check if the dataset already exists on your Google Drive; copy from Drive if found, otherwise download.
2. **Kaggle**: Check if the dataset is attached under `/kaggle/input/` (add it via the Data tab first); symlink or extract directly into the working directory.

In [ ]:
import shutil
import zipfile
from pathlib import Path
import gdown
import yaml

# Google Drive public file ID for dataset zip
GDRIVE_FILE_ID = "1rgVUTZk3M3CzsQKSRgz4ReJho3lfJNLK"

local_zip = LOCAL_DIR / "chessred_yolo_format.zip"
dataset_target = LOCAL_DIR / "chessred_yolo_format"

# Download and extract dataset if missing
if not (dataset_target.exists() and any(dataset_target.iterdir())):
    if ZIP_DRIVE_PATH and ZIP_DRIVE_PATH.exists():
        print(f"Copying dataset zip from Google Drive: {ZIP_DRIVE_PATH}")
        shutil.copy(ZIP_DRIVE_PATH, local_zip)
    else:
        print("Downloading dataset zip from Google Drive via gdown...")
        gdown.download(id=GDRIVE_FILE_ID, output=str(local_zip), quiet=False)

    print("Extracting YOLO dataset archive...")
    with zipfile.ZipFile(local_zip, "r") as zr:
        zr.extractall(LOCAL_DIR)
    local_zip.unlink(missing_ok=True)
    print("Dataset extraction completed successfully.")

# Update data.yaml path attribute to absolute local path
yaml_path = dataset_target / "data.yaml"
if yaml_path.exists():
    with open(yaml_path, "r") as f:
        data_cfg = yaml.safe_load(f)

    data_cfg["path"] = str(dataset_target)
    with open(yaml_path, "w") as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
    print(f"Updated data.yaml path to: {dataset_target}")


annotations.json already exists.
Dataset images are already present and extracted.


## Step 3 - Run Fine-Tuning with Ultralytics API

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11m checkpoint
model = YOLO("yolo11m.pt")

# Train fine-tuning model directly inside notebook cell
results = model.train(
    data=str(dataset_target / "data.yaml"),
    project=RUNS_DIR,
    name="chess_detection_yolo11m",
    epochs=60,
    batch=16,
    imgsz=640,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    plots=True,
    exist_ok=True,
)


Successfully wrote dataset.yaml to: /kaggle/working/datasets/dataset.yaml


In [ ]:
!rm -rf /kaggle/working/runs/

In [ ]:
!rm /kaggle/working/datasets/chessred_train.cache
!rm /kaggle/working/datasets/chessred_val.cache

In [12]:
# Thay 'ten_file_nen.zip' bằng tên bạn muốn lưu
# Thay '/kaggle/working/ten_thu_muc_cua_ban' bằng đường dẫn tới folder bạn muốn tải
!zip -r ok.zip /kaggle/working/runs/chess_detection_yolo11m-2


  adding: kaggle/working/runs/chess_detection_yolo11m-2/ (stored 0%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/val_batch2_labels.jpg (deflated 3%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/BoxPR_curve.png (deflated 17%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/train_batch1.jpg (deflated 7%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/BoxR_curve.png (deflated 20%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/train_batch2.jpg (deflated 5%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/args.yaml (deflated 54%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/val_batch2_pred.jpg (deflated 3%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/val_batch0_labels.jpg (deflated 2%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/val_batch0_pred.jpg (deflated 2%)
  adding: kaggle/working/runs/chess_detection_yolo11m-2/val_batch1_labels.jpg (deflated 2%)
  adding: kaggle/working/runs/chess_detection_y